# Laboratorio: Fourier, frecuencias y DCT

En este laboratorio veremos una misma idea en tres contextos: coordenadas complejas, señales discretas e imágenes. Al finalizar podrás:

- usar conjugación, módulo y la fórmula de Euler;
- aproximar una onda periódica con sumas parciales de Fourier;
- calcular y verificar una DFT;
- comprobar Parseval y el error de una proyección en frecuencias;
- construir una DCT ortonormal y comprimir una imagen reproducible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, interact

np.set_printoptions(precision=6, suppress=True)

## 1. Complejos: solo lo necesario

Para $z=a+bi$, NumPy usa la notación `a + b*j`. Verificaremos $z\overline z=|z|^2$ y $e^{i\theta}=\cos\theta+i\sin\theta$.

In [ ]:
z = 1 + np.sqrt(3) * 1j
r = np.abs(z)
theta = np.angle(z)
z_polar = r * np.exp(1j * theta)

print("z =", z)
print("conjugado =", np.conj(z))
print("módulo =", r)
print("argumento =", theta, "rad =", theta / np.pi, "π")
print("reconstrucción polar =", z_polar)
print("z * conjugado(z) =", z * np.conj(z))

assert np.allclose(z_polar, z)
assert np.allclose(z * np.conj(z), np.abs(z)**2)

In [ ]:
angulos = np.linspace(0, 2 * np.pi, 200)
circulo = np.exp(1j * angulos)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(circulo.real, circulo.imag, color="#14578c")
ax.arrow(0, 0, (z/r).real, (z/r).imag, width=0.015, color="#be2d37", length_includes_head=True)
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(0, color="gray", linewidth=0.8)
ax.set_aspect("equal")
ax.set_xlabel("parte real")
ax.set_ylabel("parte imaginaria")
ax.set_title(r"$e^{i\theta}$ sobre el círculo unitario")
ax.grid(alpha=0.2)
plt.show()

## 2. Sumas parciales de una onda cuadrada

Para $f(t)=\operatorname{sign}(\sin t)$, la suma parcial correcta es

$$S_Nf(t)=\frac4\pi\sum_{1\leq k\leq N,\ k\text{ impar}}\frac{\sin(kt)}{k}.$$

Mueve el deslizador y observa la mejora global y la sobreoscilación cerca de los saltos.

In [ ]:
def onda_cuadrada(t):
    return np.where(np.sin(t) >= 0, 1.0, -1.0)

def suma_fourier_cuadrada(t, N):
    resultado = np.zeros_like(t, dtype=float)
    for k in range(1, N + 1, 2):
        resultado += np.sin(k * t) / k
    return (4 / np.pi) * resultado

def graficar_onda_cuadrada(N=9):
    t = np.linspace(-2 * np.pi, 2 * np.pi, 1600)
    original = onda_cuadrada(t)
    aproximacion = suma_fourier_cuadrada(t, N)
    error_medio = np.mean((original - aproximacion)**2)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(t, original, color="#14578c", linewidth=2, label="onda cuadrada")
    ax.plot(t, aproximacion, color="#be2d37", label=f"suma parcial, N={N}")
    ax.set_title(f"Error cuadrático medio muestreado: {error_medio:.5f}")
    ax.set_xlabel("t")
    ax.set_ylabel("amplitud")
    ax.grid(alpha=0.2)
    ax.legend()
    plt.show()

interact(graficar_onda_cuadrada, N=IntSlider(min=1, max=49, step=2, value=9, description="N"))

## 3. DFT como cambio a coordenadas de frecuencia

Construiremos una señal con frecuencias $3$ y $9$. Con nuestra normalización, `np.fft.fft(x) / M` contiene los coeficientes $\widehat x_k$.

In [ ]:
M = 64
n = np.arange(M)
senal = 0.8 * np.sin(2 * np.pi * 3 * n / M) + 0.35 * np.cos(2 * np.pi * 9 * n / M)

coef_directos = np.array([
    np.mean(senal * np.exp(-2j * np.pi * k * n / M))
    for k in range(M)
])
coef_fft = np.fft.fft(senal) / M

print("Máxima diferencia entre fórmula y FFT:", np.max(np.abs(coef_directos - coef_fft)))
assert np.allclose(coef_directos, coef_fft)

In [ ]:
frecuencias = np.fft.fftfreq(M) * M
orden = np.argsort(frecuencias)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(n, senal, marker="o", markersize=3, color="#14578c")
axes[0].set_title("Señal muestreada")
axes[0].set_xlabel("índice n")
axes[0].grid(alpha=0.2)
axes[1].stem(frecuencias[orden], np.abs(coef_fft[orden]), basefmt=" ")
axes[1].set_xlim(-15, 15)
axes[1].set_title("Magnitud de los coeficientes DFT")
axes[1].set_xlabel("frecuencia k")
axes[1].grid(alpha=0.2)
plt.tight_layout()
plt.show()

### Reconstrucción y Parseval

La señal se reconstruye con la base exponencial. Parseval afirma que la energía promedio en muestras coincide con la suma de energías de los coeficientes.

In [ ]:
senal_reconstruida = np.fft.ifft(M * coef_fft).real
energia_muestras = np.mean(np.abs(senal)**2)
energia_coeficientes = np.sum(np.abs(coef_fft)**2)

print("Error máximo de reconstrucción:", np.max(np.abs(senal - senal_reconstruida)))
print("Energía en muestras:", energia_muestras)
print("Energía en frecuencias:", energia_coeficientes)

assert np.allclose(senal, senal_reconstruida)
assert np.allclose(energia_muestras, energia_coeficientes)

### Proyección sobre frecuencias bajas

Para conservar una reconstrucción real mantendremos juntas las frecuencias $k$ y $-k$. La energía descartada debe coincidir con el error cuadrático promedio.

In [ ]:
def proyectar_frecuencias_bajas(coeficientes, K):
    frecs = np.fft.fftfreq(len(coeficientes)) * len(coeficientes)
    mascara = np.abs(frecs) <= K
    truncados = np.where(mascara, coeficientes, 0)
    reconstruccion = np.fft.ifft(len(coeficientes) * truncados).real
    return reconstruccion, truncados

for K in [2, 3, 5, 9]:
    aproximacion, truncados = proyectar_frecuencias_bajas(coef_fft, K)
    error = np.mean((senal - aproximacion)**2)
    energia_descartada = np.sum(np.abs(coef_fft - truncados)**2)
    print(f"K={K:2d}: error={error:.8f}, energía descartada={energia_descartada:.8f}")
    assert np.allclose(error, energia_descartada)

## 4. Construcción de la matriz DCT

Implementaremos directamente la matriz DCT-II ortonormal. No necesitamos una biblioteca adicional: la transformación será una multiplicación por una matriz ortogonal.

In [ ]:
def matriz_dct(N):
    n = np.arange(N)
    k = np.arange(N)[:, None]
    C = np.sqrt(2 / N) * np.cos(np.pi * (2 * n + 1) * k / (2 * N))
    C[0, :] = 1 / np.sqrt(N)
    return C

C8 = matriz_dct(8)
print("C para N=8:\n", C8)
print("Máximo error en C @ C.T - I:", np.max(np.abs(C8 @ C8.T - np.eye(8))))
assert np.allclose(C8 @ C8.T, np.eye(8))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for k in range(5):
    ax.plot(np.arange(8), C8[k], marker="o", label=f"patrón k={k}")
ax.set_xlabel("posición n")
ax.set_ylabel("valor de la base")
ax.set_title("Primeros patrones de la DCT")
ax.grid(alpha=0.2)
ax.legend(ncol=2)
plt.show()

## 5. Una imagen reproducible

Para que el laboratorio funcione sin descargar archivos, construiremos una imagen sintética de $64\times64$ píxeles con gradiente, una región circular y un rectángulo.

In [ ]:
N_img = 64
x = np.linspace(0, 1, N_img)
Xg, Yg = np.meshgrid(x, x)
imagen = 35 + 95 * Xg + 45 * Yg
imagen += 65 * ((Xg - 0.35)**2 + (Yg - 0.35)**2 < 0.13**2)
imagen -= 45 * ((Xg > 0.58) & (Xg < 0.86) & (Yg > 0.58) & (Yg < 0.82))
imagen = np.clip(imagen, 0, 255)

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(imagen, cmap="gray", vmin=0, vmax=255)
ax.set_title("Imagen sintética original")
ax.axis("off")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.show()

## 6. DCT bidimensional y reconstrucción

Si $C$ es ortogonal, $B=CAC^T$ contiene las coordenadas DCT y $A=C^TBC$ reconstruye exactamente la imagen.

In [ ]:
C = matriz_dct(N_img)
coef_dct = C @ imagen @ C.T
imagen_exacta = C.T @ coef_dct @ C

print("Error máximo de reconstrucción:", np.max(np.abs(imagen - imagen_exacta)))
print("Norma original:", np.linalg.norm(imagen))
print("Norma de coeficientes:", np.linalg.norm(coef_dct))

assert np.allclose(imagen, imagen_exacta)
assert np.allclose(np.linalg.norm(imagen), np.linalg.norm(coef_dct))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(imagen, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Imagen")
axes[0].axis("off")
mapa = axes[1].imshow(np.log1p(np.abs(coef_dct)), cmap="magma")
axes[1].set_title("log(1 + |coeficientes DCT|)")
axes[1].set_xlabel("frecuencia horizontal")
axes[1].set_ylabel("frecuencia vertical")
plt.colorbar(mapa, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

### Conservar un bloque de bajas frecuencias

Conservar las primeras $K\times K$ coordenadas es proyectar sobre los patrones de menor frecuencia horizontal y vertical.

In [ ]:
def reconstruir_bajas_frecuencias(K):
    truncados = np.zeros_like(coef_dct)
    truncados[:K, :K] = coef_dct[:K, :K]
    reconstruida = C.T @ truncados @ C
    error_relativo = np.linalg.norm(imagen - reconstruida) / np.linalg.norm(imagen)
    energia = np.linalg.norm(truncados)**2 / np.linalg.norm(coef_dct)**2
    return reconstruida, truncados, error_relativo, energia

def mostrar_compresion(K=8):
    reconstruida, truncados, error, energia = reconstruir_bajas_frecuencias(K)
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].imshow(imagen, cmap="gray", vmin=0, vmax=255)
    axes[0].set_title("Original")
    axes[1].imshow(reconstruida, cmap="gray", vmin=0, vmax=255)
    axes[1].set_title(f"K={K}: {K*K} coeficientes\nenergía={energia:.3%}, error={error:.3%}")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

interact(mostrar_compresion, K=IntSlider(min=1, max=32, step=1, value=8, description="K"))

In [ ]:
# La energía descartada coincide con el error cuadrático de Frobenius.
K = 8
reconstruida, truncados, _, _ = reconstruir_bajas_frecuencias(K)
error_cuadrado = np.linalg.norm(imagen - reconstruida)**2
energia_descartada = np.linalg.norm(coef_dct - truncados)**2
print("Error cuadrático:", error_cuadrado)
print("Energía descartada:", energia_descartada)
assert np.allclose(error_cuadrado, energia_descartada)

### Bajas frecuencias frente a coeficientes de mayor magnitud

Para un número fijo de coordenadas, conservar las de mayor magnitud minimiza el error entre todas las selecciones de coordenadas DCT. Sin embargo, describir sus posiciones también tiene un costo; los formatos prácticos aprovechan estructuras y cuantización.

In [ ]:
K = 8
cantidad = K * K
_, bloque_bajo, _, _ = reconstruir_bajas_frecuencias(K)

indices_mayores = np.argpartition(np.abs(coef_dct).ravel(), -cantidad)[-cantidad:]
mayores = np.zeros_like(coef_dct).ravel()
mayores[indices_mayores] = coef_dct.ravel()[indices_mayores]
mayores = mayores.reshape(coef_dct.shape)

imagen_baja = C.T @ bloque_bajo @ C
imagen_mayores = C.T @ mayores @ C
error_bajo = np.linalg.norm(imagen - imagen_baja)
error_mayores = np.linalg.norm(imagen - imagen_mayores)

print(f"Error con bloque {K}x{K}: {error_bajo:.6f}")
print(f"Error con los {cantidad} mayores: {error_mayores:.6f}")
assert error_mayores <= error_bajo + 1e-10

## 7. Actividades

1. Cambia las frecuencias $3$ y $9$ de la señal. Predice dónde aparecerán los picos de la DFT antes de ejecutar.
2. Añade ruido a la señal y conserva solo frecuencias bajas. Compara el error con la apariencia visual.
3. Construye la matriz DFT para $M=8$ y verifica directamente la ortonormalidad con el producto interno promedio.
4. Cambia la imagen sintética agregando líneas finas. ¿Qué ocurre con los coeficientes de alta frecuencia?
5. Compara $K=4,8,16,32$ mediante porcentaje de energía y error relativo.
6. Explica por qué anular coeficientes DCT es una proyección, mientras que redondearlos es una operación diferente.